In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path(".")

files = [
    ("A_7_5g",  ROOT/"APhMm_7_5g"/"APhMm_7_5g_clean.csv"),
    ("A_15g",   ROOT/"APhMm_15g"/"APhMm_15g_clean.csv"),
    ("A_18g",   ROOT/"APhMm_18g"/"APhMm_18g_clean.csv"),
    ("A_22_5g", ROOT/"APhMm_22_5g"/"APhMm_22_5g_clean.csv"),
]

def load(path):
    df = pd.read_csv(path)
    # Arreglar por si quedaron nombres raros:
    df = df.rename(columns={c.lower().strip():c for c in df.columns})
    if not {"nm","A"}.issubset(df.columns):
        # Si venía como x,y entonces lo corregimos
        c0, c1 = df.columns[:2]
        df = df.rename(columns={c0:"nm", c1:"A"})
    df["nm"] = pd.to_numeric(df["nm"], errors="coerce")
    df["A"]  = pd.to_numeric(df["A"], errors="coerce")
    return df.dropna()[["nm","A"]].sort_values("nm").drop_duplicates()

# ---- MATRIZ SIN INTERPOLAR (INTERSECCIÓN EXACTA) ----

mat = load(files[0][1]).rename(columns={"A": files[0][0]})

for label, path in files[1:]:
    df = load(path).rename(columns={"A": label})
    mat = mat.merge(df, on="nm", how="inner")   # SOLO nm que existen en todos

mat = mat.sort_values("nm").reset_index(drop=True)

out = ROOT / "APhMm_matrix.csv"
mat.to_csv(out, index=False)

print("✔ Matriz generada correctamente:")
print(out)
display(mat.head())

✔ Matriz generada correctamente:
APhMm_matrix.csv


,nm,A_7_5g,A_15g,A_18g,A_22_5g
0,231.0,0.751,1.513,1.453,2.434
1,231.5,0.743,1.494,1.437,2.386
2,232.0,0.735,1.476,1.421,2.338
3,232.5,0.726,1.459,1.404,2.293
4,233.0,0.718,1.441,1.386,2.249
